In [ ]:
# ======================================================
# Notebook: Drug Combination Optimisation (Hyperparameter tuning)
# Maximise (- side effects)
# ======================================================

import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

# Load data
X = np.load("/mnt/data/initial_inputs.npy")      # (N,3)
y_raw = np.load("/mnt/data/initial_outputs.npy") # (N,)

# Transform objective
y = -y_raw

# Hyperparameter tuning
param_dist = {
    "n_estimators": [50,100,200,300],
    "max_depth": [None,4,6,10],
    "min_samples_split": [2,5,10],
    "min_samples_leaf": [1,2,4]
}

search = RandomizedSearchCV(
    RandomForestRegressor(),
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    random_state=42
)

search.fit(X, y)
model = search.best_estimator_

print("Best params:", search.best_params_)

# Candidate grid
bounds = [(X[:,i].min(), X[:,i].max()) for i in range(3)]
grid = [np.linspace(b[0], b[1], 20) for b in bounds]
X_grid = np.array(np.meshgrid(*grid)).T.reshape(-1,3)

# Ensemble predictions for exploration/exploitation
preds = np.array([tree.predict(X_grid) for tree in model.estimators_])
mean_pred = preds.mean(axis=0)
uncertainty = preds.std(axis=0)

# Acquisition
acquisition = mean_pred + 0.5 * uncertainty

# Select next (10,3)
top_idx = np.argsort(acquisition)[-10:]
next_points = X_grid[top_idx]

print("Next (10,3) compound combinations:")
print(next_points)